# Task 3 - Running a model locally and optimising it

**Setup:** Ollama running inside Colab on the T4 GPU, serving **Qwen2.5-3B-Instruct** in two settings:
- **fp16** (16-bit, the "unquantised" reference)
- **q4_K_M** (4-bit quantised)

We measure tokens/second and GPU memory, compare output quality on the same 10 prompts, then do the KV-cache and 20-user serving calculations.

**Groq's role:** Groq does **not** run models on your machine, so it cannot be the thing you benchmark. It is used here only as an LLM judge (Step 5) and, optionally, as a hosted-speed reference (Step 6).

Runtime: **T4 GPU** required. Key in Colab Secrets as `GROQ_API_KEY` (only needed for the judge).

In [34]:
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv

name, memory.total [MiB], memory.used [MiB]
Tesla T4, 15360 MiB, 2193 MiB


In [35]:
!apt-get -qq install -y zstd pciutils lshw > /dev/null 2>&1
!curl -fsSL https://ollama.com/install.sh | sh

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> NVIDIA GPU installed.
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [36]:
import subprocess, time, requests, os, json
OLLAMA = "http://127.0.0.1:11434"
env = dict(os.environ, OLLAMA_HOST="127.0.0.1:11434")
server = subprocess.Popen(["ollama", "serve"], stdout=open("/content/ollama.log", "w"),
                          stderr=subprocess.STDOUT, env=env)
up = False
for _ in range(60):
    try:
        requests.get(OLLAMA, timeout=1)
        up = True
        break
    except Exception:
        time.sleep(1)
print("Ollama is up" if up else "Ollama did NOT start - read /content/ollama.log")

Ollama is up


In [37]:
# ~6 GB and ~2 GB downloads
!ollama pull qwen2.5:3b-instruct-fp16
!ollama pull qwen2.5:3b-instruct-q4_K_M
tags = requests.get(f"{OLLAMA}/api/tags").json()["models"]
for m in tags:
    print(m["name"], f"{m['size']/1e9:.2f} GB on disk")



qwen2.5:3b-instruct-q4_K_M 1.93 GB on disk
qwen2.5:3b-instruct-fp16 6.18 GB on disk


## Step 1 - Benchmark helpers
- **Tokens/s (decode)** = `eval_count / eval_duration` from Ollama's own response (generation speed, excludes prompt processing).
- **Memory** = GPU memory used by the loaded model: `nvidia-smi` before vs after loading, plus Ollama's `size_vram`.
- Same settings for both: temperature 0, seed 42, context 4096, max 256 new tokens.

In [38]:
SETTINGS = {"fp16": "qwen2.5:3b-instruct-fp16", "q4_K_M": "qwen2.5:3b-instruct-q4_K_M"}

def gpu_mem_mb():
    out = subprocess.check_output(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"]).decode()
    return int(out.strip().split("\n")[0])

def loaded_models():
    return requests.get(f"{OLLAMA}/api/ps").json().get("models", [])

def unload_all():
    for m in loaded_models():
        requests.post(f"{OLLAMA}/api/generate", json={"model": m["name"], "keep_alive": 0})
    time.sleep(5)

def generate(model, prompt, num_predict=256):
    r = requests.post(f"{OLLAMA}/api/generate", timeout=900, json={
        "model": model, "prompt": prompt, "stream": False, "keep_alive": "30m",
        "options": {"temperature": 0, "seed": 42, "num_predict": num_predict, "num_ctx": 4096}})
    r.raise_for_status()
    return r.json()

PROMPTS = [
    "Explain in 3 sentences why the sky is blue.",
    "A train leaves at 2:15 PM and travels 180 km at 60 km/h. At what time does it arrive? Show your calculation.",
    "Write a Python function that returns the n-th Fibonacci number iteratively, with a docstring.",
    "Summarize in exactly two sentences: 'The city council voted 7-2 on Tuesday to approve a new network of protected bike lanes covering 40 km of roads. Supporters said it will cut traffic injuries, while opponents argued it will remove parking and hurt local shops. Construction begins in March and is expected to take 18 months.'",
    "Translate to French: 'Could you tell me where the nearest train station is?'",
    "List five differences between a process and a thread as a markdown table.",
    "Return ONLY valid JSON with keys name, age, city for this sentence: Anita is 29 and lives in Pune.",
    "What is 17 x 24? Give just the number.",
    "Write a four-line poem about autumn rain.",
    "Give three pros and three cons of microservices, in under 120 words.",
]
assert len(PROMPTS) == 10

## Step 2 - Run both settings (tokens/s and memory)

In [39]:
import numpy as np, pandas as pd
disk = {m["name"]: m["size"] / 1e9 for m in requests.get(f"{OLLAMA}/api/tags").json()["models"]}
speed_rows, outputs = [], {}

for name, tag in SETTINGS.items():
    unload_all()
    mem_before = gpu_mem_mb()
    t0 = time.time()
    generate(tag, "Hello", 8)                      # loads model into GPU
    load_s = time.time() - t0
    mem_after = gpu_mem_mb()
    lm = loaded_models()
    vram = lm[0].get("size_vram", 0) / 1e9 if lm else float("nan")

    tps, pps, outs = [], [], []
    for p in PROMPTS:
        r = generate(tag, p)
        outs.append(r["response"].strip())
        tps.append(r["eval_count"] / (r["eval_duration"] / 1e9))
        if r.get("prompt_eval_duration"):
            pps.append(r["prompt_eval_count"] / (r["prompt_eval_duration"] / 1e9))
    outputs[name] = outs
    mem_peak = gpu_mem_mb()
    speed_rows.append({
        "setting": name, "disk_GB": round(disk.get(tag, float("nan")), 2),
        "GPU_mem_loaded_MB": mem_after - mem_before, "ollama_size_vram_GB": round(vram, 2),
        "GPU_mem_after_run_MB": mem_peak - mem_before,
        "decode_tok/s_mean": round(np.mean(tps), 1), "decode_tok/s_std": round(np.std(tps), 1),
        "prompt_tok/s_mean": round(np.mean(pps), 1) if pps else None,
        "load_time_s": round(load_s, 1)})
    print(name, "done")

speed = pd.DataFrame(speed_rows)
speed

fp16 done
q4_K_M done


,setting,disk_GB,GPU_mem_loaded_MB,ollama_size_vram_GB,GPU_mem_after_run_MB,decode_tok/s_mean,decode_tok/s_std,prompt_tok/s_mean,load_time_s
0,fp16,6.18,6255,6.41,6255,38.0,3.3,1122.4,16.0
1,q4_K_M,1.93,2191,2.16,2193,75.7,8.8,1661.7,3.3


## Step 3 - Output quality on the same 10 prompts
Two layers of evidence:
1. **Automatic checks** on prompts that have a verifiable answer (train arrival time = 5:15 PM, valid JSON, 17 x 24 = 408).
2. **LLM judge (Groq Llama-3.3-70B)** scores each pair blind (random A/B order) from 1-5. Treat this as supporting evidence only - **read the outputs yourself** (next cell) and write your own observations.

In [40]:
def check_train(o): return "5:15" in o or "17:15" in o
def check_json(o):
    s, e = o.find("{"), o.rfind("}")
    try:
        d = json.loads(o[s:e + 1])
        return set(d) >= {"name", "age", "city"} and int(d["age"]) == 29 and "pune" in str(d["city"]).lower()
    except Exception:
        return False
def check_mult(o): return "408" in o

AUTO = {1: check_train, 6: check_json, 7: check_mult}
rows = []
for idx, fn in AUTO.items():
    rows.append({"prompt": idx + 1, **{s: fn(outputs[s][idx]) for s in SETTINGS}})
print(pd.DataFrame(rows).to_string(index=False))

 prompt  fp16  q4_K_M
      2  True    True
      7  True    True
      8  True    True


In [41]:
# Read both answers side by side (this is the raw material for your observations)
for i, p in enumerate(PROMPTS):
    print("=" * 100)
    print(f"PROMPT {i+1}: {p[:150]}")
    for s in SETTINGS:
        print(f"\n--- {s} ---\n{outputs[s][i][:600]}")

PROMPT 1: Explain in 3 sentences why the sky is blue.

--- fp16 ---
The sky appears blue because of a phenomenon called Rayleigh scattering. When sunlight enters the Earth's atmosphere, it collides with molecules and small particles in the air. Shorter wavelength light, such as blue and violet, is scattered more than longer wavelength light like red and orange. This blue light is scattered in all directions, which means it reaches our eyes from all parts of the sky, giving the sky its blue color.

--- q4_K_M ---
The sky appears blue because of a phenomenon called Rayleigh scattering. When sunlight enters the Earth's atmosphere, it interacts with molecules and small particles in the air. Shorter wavelength light, such as blue and violet light, is scattered more than longer wavelength light like red and yellow light. This is why we see a predominantly blue sky during the day.
PROMPT 2: A train leaves at 2:15 PM and travels 180 km at 60 km/h. At what time does it arrive? Show your calcula

In [42]:
!pip install -U groq

In [43]:
import random
try:
    from google.colab import userdata
    os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
except Exception:
    import getpass
    os.environ["GROQ_API_KEY"] = getpass.getpass("Paste your Groq API key: ")
from groq import Groq
client = Groq()
JUDGE_MODEL = "openai/gpt-oss-120b"

JUDGE_SYS = ("You are a strict grader. You see a PROMPT and two answers, A and B. Score each answer from 1 to 5 for "
             "correctness, instruction-following and clarity. "
             'Respond ONLY with JSON: {"A": <int>, "B": <int>, "note": "<one sentence>"}')

rng = random.Random(0)
judged = []
for i, p in enumerate(PROMPTS):
    flip = rng.random() < 0.5
    a_name, b_name = ("q4_K_M", "fp16") if flip else ("fp16", "q4_K_M")
    user = f"PROMPT:\n{p}\n\nANSWER A:\n{outputs[a_name][i]}\n\nANSWER B:\n{outputs[b_name][i]}"
    scores = None
    for attempt in range(4):
        try:
            resp = client.chat.completions.create(
                model=JUDGE_MODEL, temperature=0, response_format={"type": "json_object"},
                messages=[{"role": "system", "content": JUDGE_SYS}, {"role": "user", "content": user}])
            scores = json.loads(resp.choices[0].message.content)
            break
        except Exception as e:
            print("retry:", str(e)[:100]); time.sleep(10 * (attempt + 1))
    if scores:
        judged.append({"prompt": i + 1, a_name: scores["A"], b_name: scores["B"], "note": scores.get("note", "")})
    time.sleep(1)

if judged:
    jdf = pd.DataFrame(judged)
    print(jdf.to_string(index=False))
    print("\nMean judge score:\n", jdf[["fp16", "q4_K_M"]].mean().round(2).to_dict())
else:
    print("No results were evaluated because all API requests failed. Please check your API key and model access.")

 prompt  fp16  q4_K_M                                                                                                                              note
      1     3       3                                                 Both answers are correct and clear but exceed the requested three-sentence limit.
      2     5       5                                                                  Both answers correctly compute and clearly present the solution.
      3     5       4   Both answers meet the prompt, but B provides clearer indexing, examples, and a more complete docstring, earning a higher score.
      4     2       2                 Both answers are single-sentence summaries missing the vote count, thus not meeting the two‑sentence requirement.
      5     5       5                                                Both answers correctly translate the sentence, follow instructions, and are clear.
      6     5       2   Answer B correctly follows the prompt with a clear, well‑formatt

## Step 5 - KV cache and quantisation: numbers
**KV cache per token** = 2 (K and V) x layers x KV-heads x head_dim x bytes-per-value.
It grows linearly with context length and with the number of simultaneous users, and often exceeds the memory used by the weights at high concurrency.

**Quantisation** stores weights in fewer bits: 16-bit = 2 bytes/param, 8-bit = 1 byte, 4-bit = ~0.5 byte (+ small scale/zero-point overhead). Memory for weights drops about 4x for 4-bit; decode speed often rises too because token generation is memory-bandwidth-bound (fewer bytes to read per token).

In [45]:
def kv_bytes_per_token(layers, kv_heads, head_dim, bytes_per_val=2):
    return 2 * layers * kv_heads * head_dim * bytes_per_val

MODELS = {   # architecture facts from the public model configs
    "Qwen2.5-3B  (36 layers, 2 KV heads, d=128)": (36, 2, 128),
    "Llama-3.1-8B (32 layers, 8 KV heads, d=128)": (32, 8, 128),
}
for name, (L, H, D) in MODELS.items():
    b = kv_bytes_per_token(L, H, D)
    print(f"{name}: {b/1024:.0f} KB/token (fp16) -> {b*4096/1e9:.2f} GB for ONE 4096-token sequence")

# weights memory by precision
def weights_gb(params_b, bits, overhead=1.0):
    return params_b * 1e9 * bits / 8 * overhead / 1e9
print()
for bits, ov, label in [(16, 1.0, "fp16"), (8, 1.0, "int8"), (4, 1.1, "4-bit (+~10% scales)")]:
    print(f"3B model {label:>22}: {weights_gb(3.09, bits, ov):.2f} GB   |   8B model: {weights_gb(8.03, bits, ov):.2f} GB")

Qwen2.5-3B  (36 layers, 2 KV heads, d=128): 36 KB/token (fp16) -> 0.15 GB for ONE 4096-token sequence
Llama-3.1-8B (32 layers, 8 KV heads, d=128): 128 KB/token (fp16) -> 0.54 GB for ONE 4096-token sequence

3B model                   fp16: 6.18 GB   |   8B model: 16.06 GB
3B model                   int8: 3.09 GB   |   8B model: 8.03 GB
3B model   4-bit (+~10% scales): 1.70 GB   |   8B model: 4.42 GB


## Step 6 - Plan: serve 20 concurrent users on ONE 24 GB GPU
**Assumptions (state these in your report; change them in the cell and re-run):**
- Model: Llama-3.1-8B-class, **4-bit AWQ** (~5.7 GB, since embeddings stay 16-bit)
- 24 GB GPU (RTX 4090 / 3090 class), memory bandwidth ~1000 GB/s
- Each user: up to **4096 tokens** of context (prompt + generated)
- Serving engine: **vLLM** (continuous batching + PagedAttention, so KV memory is allocated in blocks as needed, not reserved at max length)
- ~2 GB reserved for CUDA context, activations and CUDA graphs; 90% of memory usable by vLLM

**Launch (on the GPU machine):**
`vllm serve hugging-quants/Meta-Llama-3.1-8B-Instruct-AWQ-INT4 --quantization awq --max-model-len 4096 --max-num-seqs 20 --gpu-memory-utilization 0.90 --enable-prefix-caching`
(add `--kv-cache-dtype fp8` to halve KV memory if you need more users or longer context)

In [46]:
GPU_GB, USABLE = 24, 0.90
WEIGHTS_4BIT, WEIGHTS_FP16, OVERHEAD = 5.7, 16.06, 2.0
USERS, CTX = 20, 4096
LAYERS, KV_HEADS, HEAD_DIM = 32, 8, 128
BANDWIDTH_GBPS, EFFICIENCY = 1000, 0.6

kv_tok = kv_bytes_per_token(LAYERS, KV_HEADS, HEAD_DIM, 2)          # fp16 KV
kv_total = USERS * CTX * kv_tok / 1e9
kv_total_fp8 = kv_total / 2
budget = GPU_GB * USABLE

print(f"KV per token: {kv_tok/1024:.0f} KB | KV for {USERS} users x {CTX} tokens: {kv_total:.1f} GB (fp16) / {kv_total_fp8:.1f} GB (fp8)")
for label, w, kv in [("4-bit AWQ + fp16 KV", WEIGHTS_4BIT, kv_total),
                     ("4-bit AWQ + fp8 KV ", WEIGHTS_4BIT, kv_total_fp8),
                     ("fp16 weights + fp16 KV", WEIGHTS_FP16, kv_total),
                     ("fp16 weights + fp8 KV ", WEIGHTS_FP16, kv_total_fp8)]:
    need = w + kv + OVERHEAD
    print(f"{label:<24} need {need:5.1f} GB vs budget {budget:.1f} GB -> {'FITS' if need <= budget else 'DOES NOT FIT'}")

max_users = (budget - WEIGHTS_4BIT - OVERHEAD) * 1e9 / (CTX * kv_tok)
print(f"\nMax concurrent {CTX}-token users with 4-bit weights + fp16 KV: ~{max_users:.0f}")

# decode speed estimate: memory-bandwidth bound. Each step reads all weights + all active KV once.
step_bytes = (WEIGHTS_4BIT + kv_total) * 1e9
step_s = step_bytes / (BANDWIDTH_GBPS * 1e9 * EFFICIENCY)
print(f"Decode step ~{step_s*1000:.0f} ms (all {USERS} users advance one token) -> ~{1/step_s:.0f} tokens/s PER USER, ~{USERS/step_s:.0f} tokens/s total")
print("Prefill is compute-bound: long prompts from many users arriving together will queue; mitigate with chunked prefill and prefix caching for the shared system prompt.")

KV per token: 128 KB | KV for 20 users x 4096 tokens: 10.7 GB (fp16) / 5.4 GB (fp8)
4-bit AWQ + fp16 KV      need  18.4 GB vs budget 21.6 GB -> FITS
4-bit AWQ + fp8 KV       need  13.1 GB vs budget 21.6 GB -> FITS
fp16 weights + fp16 KV   need  28.8 GB vs budget 21.6 GB -> DOES NOT FIT
fp16 weights + fp8 KV    need  23.4 GB vs budget 21.6 GB -> DOES NOT FIT

Max concurrent 4096-token users with 4-bit weights + fp16 KV: ~26
Decode step ~27 ms (all 20 users advance one token) -> ~37 tokens/s PER USER, ~730 tokens/s total
Prefill is compute-bound: long prompts from many users arriving together will queue; mitigate with chunked prefill and prefix caching for the shared system prompt.


## Step 7 - Write these in your own words (prompts to structure your thinking)
**What the KV cache is:** during generation each new token needs the Keys and Values of all previous tokens in attention. Recomputing them every step would be quadratic work; instead we store them (the cache) and compute only the new token's K/V. Cost: memory grows with context length x users x layers. Use your Step 5 numbers.

**Why quantisation saves memory:** weights are stored with fewer bits per parameter (16 -> 4 bits is ~4x smaller), using per-group scales to map back to approximate values. Compare your measured fp16 vs q4_K_M memory, mention the small quality loss you observed (or did not), and that quantised weights also speed up decoding because generation is limited by memory bandwidth, not compute. Note that quantising weights does not shrink the KV cache unless you also quantise the KV cache.

In [47]:
# STEP 7 helper: pulls YOUR measured numbers together so you can write the two explanations.
# Run Steps 2, 3 and 6 first (it uses `speed`, `outputs`, `jdf`, `kv_bytes_per_token`).
import numpy as np

print("=" * 70)
print("A. KV CACHE (use these in your KV-cache explanation)")
print("=" * 70)
kv3 = kv_bytes_per_token(36, 2, 128)      # Qwen2.5-3B, the model you actually ran
kv8 = kv_bytes_per_token(32, 8, 128)      # Llama-3.1-8B, the model in the 20-user plan
print(f"Qwen2.5-3B : {kv3/1024:.0f} KB per token -> {kv3*4096/1e6:.0f} MB for one 4096-token chat")
print(f"   (Ollama reserved about this much on top of the weights, because we set num_ctx=4096)")
print(f"Llama-3.1-8B: {kv8/1024:.0f} KB per token -> {kv8*4096/1e9:.2f} GB per 4096-token user")
print(f"   20 users x 4096 tokens = {20*4096*kv8/1e9:.1f} GB of KV cache (fp16), "
      f"vs ~5.7 GB for the 4-bit weights -> the cache is about {20*4096*kv8/1e9/5.7:.1f}x the weights")
print(f"   with an fp8 KV cache: {20*4096*kv8/2e9:.1f} GB")

print()
print("=" * 70)
print("B. QUANTISATION (use these in your quantisation explanation)")
print("=" * 70)
try:
    s = speed.set_index("setting")
    params_b = 3.09
    for k in ["fp16", "q4_K_M"]:
        bits = s.loc[k, "disk_GB"] * 8 / params_b
        print(f"{k:7}: file {s.loc[k,'disk_GB']:.2f} GB (= {bits:.1f} bits per weight on average) | "
              f"GPU memory loaded {s.loc[k,'GPU_mem_loaded_MB']:.0f} MB | decode {s.loc[k,'decode_tok/s_mean']} tok/s")
    f, q = s.loc["fp16"], s.loc["q4_K_M"]
    print(f"\nMemory: fp16 uses {f['GPU_mem_loaded_MB']/q['GPU_mem_loaded_MB']:.1f}x more GPU memory than q4_K_M "
          f"(file size ratio {f['disk_GB']/q['disk_GB']:.1f}x)")
    print(f"Speed : q4_K_M decodes {q['decode_tok/s_mean']/f['decode_tok/s_mean']:.1f}x faster than fp16")
    print(f"Bandwidth check (T4 has ~320 GB/s): fp16 ceiling = 320/{f['disk_GB']:.1f} = {320/f['disk_GB']:.0f} tok/s, "
          f"q4 ceiling = 320/{q['disk_GB']:.1f} = {320/q['disk_GB']:.0f} tok/s. "
          f"Measured speeds should sit below these ceilings.")
except NameError:
    print("`speed` not found. Run Step 2 first.")

try:
    print()
    for idx, fn in AUTO.items():
        print(f"Prompt {idx+1} automatic check ->", {name: fn(outputs[name][idx]) for name in SETTINGS})
    same = sum(outputs["fp16"][i].strip() == outputs["q4_K_M"][i].strip() for i in range(len(PROMPTS)))
    print(f"Identical answers: {same}/{len(PROMPTS)} prompts (greedy decoding, so differences come from the weights only)")
    print("Mean judge score:", jdf[["fp16", "q4_K_M"]].mean().round(2).to_dict())
except NameError:
    print("Quality results not found. Run Step 3 first.")

A. KV CACHE (use these in your KV-cache explanation)
Qwen2.5-3B : 36 KB per token -> 151 MB for one 4096-token chat
   (Ollama reserved about this much on top of the weights, because we set num_ctx=4096)
Llama-3.1-8B: 128 KB per token -> 0.54 GB per 4096-token user
   20 users x 4096 tokens = 10.7 GB of KV cache (fp16), vs ~5.7 GB for the 4-bit weights -> the cache is about 1.9x the weights
   with an fp8 KV cache: 5.4 GB

B. QUANTISATION (use these in your quantisation explanation)
fp16   : file 6.18 GB (= 16.0 bits per weight on average) | GPU memory loaded 6255 MB | decode 38.0 tok/s
q4_K_M : file 1.93 GB (= 5.0 bits per weight on average) | GPU memory loaded 2191 MB | decode 75.7 tok/s

Memory: fp16 uses 2.9x more GPU memory than q4_K_M (file size ratio 3.2x)
Speed : q4_K_M decodes 2.0x faster than fp16
Bandwidth check (T4 has ~320 GB/s): fp16 ceiling = 320/6.2 = 52 tok/s, q4 ceiling = 320/1.9 = 166 tok/s. Measured speeds should sit below these ceilings.

Prompt 2 automatic check -